# CET313 steel surface defect detection prototype

This notebook presents the **existing** multi-label classification and four-channel segmentation implementation. It draws completed-run results from the original CSV histories and saved figures. Every displayed historical metric is **validation**, not independent test, performance. The archived figures and examples below were copied without recomputation.

**Safety:** normal cells only read files. Training is in an opt-in cell at the end and requires editing a guard. Only the read-only cells with execution counts were run in a Jupyter kernel for this submission. The training cell and model-loading branches were not run. All displayed historical results remain labelled as archived previous-run evidence. The two original training scripts and the unchanged `src/` modules are included beside it.

This code-only copy uses the original project scripts, CSV histories, figures and selected prediction examples. No report document, model checkpoint or image dataset is bundled. The notebook was assembled from project artifacts on 7 October 2026.

## 1. Setup and evidence inventory

Run this cell from the clean submission folder. It only checks paths. Optional model evaluation requires the dependencies described in README.md. The archived evidence cells use the Python standard library and Jupyter display functions.

In [1]:
from pathlib import Path
from IPython.display import display, Markdown, Image
import csv
import statistics

ROOT = Path.cwd().resolve()
required = [ROOT / 'logs', ROOT / 'figures', ROOT / 'src', ROOT / 'train_classifier.py', ROOT / 'train_segmentation.py']
assert all(p.exists() for p in required), f'Open Jupyter in the CET313_submission folder: {ROOT}'
for p in [ROOT/'data', ROOT/'checkpoints', ROOT/'data/processed/test_split.csv']:
    print(f'{p.relative_to(ROOT)}: {"present" if p.exists() else "unavailable"}')
print('Original histories:', len(list((ROOT/'logs').glob('*.csv'))))


data: unavailable
checkpoints: unavailable
data\processed\test_split.csv: unavailable
Original histories: 8


## 2. Dataset and preprocessing

`dataset_audit.py` builds image labels from the Severstal annotations; `src/datasets/split_dataset.py` makes a multilabel-stratified 70/15/15 image-level split with seed 42. **The original data and split CSVs are absent here**, so the report's 12,568 total images, 8,797 classification training images, and class counts 536/135/3341/905 cannot be independently checked.

Classification loads RGB images, resizes to 256×256, applies training flips and brightness/contrast changes, then `A.Normalize()` and `ToTensorV2()`. Validation uses resize and normalization only. Segmentation loads RGB images and four separate RLE masks, resizes both to 256×256 (nearest neighbour for masks), scales pixels by 255, and uses flips and 90° rotation only for training. Its seed-42 80/20 split is made on **annotation rows before grouping by image**, so image overlap and incomplete grouped masks need audit with the original `data/train.csv`. The report's 5,395/1,406 segmentation train/validation image counts come from a run screenshot, not available source data.

In [2]:
# Read-only inspection of the exact preprocessing definitions used by the original scripts.
from pathlib import Path
for rel in ['src/datasets/transforms.py', 'src/datasets/classification_dataset.py',
            'src/datasets/segmentation_dataset.py', 'src/datasets/split_dataset.py']:
    print(f'{rel}: {(ROOT/rel).exists()}')


src/datasets/transforms.py: True
src/datasets/classification_dataset.py: True
src/datasets/segmentation_dataset.py: True
src/datasets/split_dataset.py: True


## 3. CNN architecture and objectives

The classifier baseline is torchvision ResNet18 with a four-logit head; CBAM-ResNet18 inserts the project's `CBAM` after `layer4` and before global pooling. Segmentation uses a ResNet18 encoder and U-Net-style skip decoder; CBAM U-Net inserts CBAM at the bottleneck. **The segmentation decoders also differ at final upsampling** (`final_up` in baseline versus bilinear interpolation in CBAM), so its comparison cannot isolate attention alone. The architecture definitions below are imported directly from the accompanying, unchanged source modules.

Classification uses the project's focal loss (alpha 0.25, gamma 2). Segmentation uses 0.5 soft Dice loss + 0.5 BCEWithLogitsLoss. Predictions use sigmoid threshold ≥0.5 for classification and >0.5 for segmentation. The latter's Dice and IoU are averaged over image–class pairs within each batch, then batches are averaged equally, including empty masks.

In [3]:
# Imports are deferred until the optional evaluation cells, so archived evidence remains readable
# on machines without PyTorch, torchvision, Albumentations, or OpenCV.
MODEL_FILES = ['src/models/resnet18_classifier.py', 'src/models/cbam_resnet18.py',
               'src/models/unet_resnet18.py', 'src/models/unet_cbam.py', 'src/models/cbam.py']
for rel in MODEL_FILES:
    print(rel, 'available' if (ROOT/rel).exists() else 'missing')


src/models/resnet18_classifier.py available
src/models/cbam_resnet18.py available
src/models/unet_resnet18.py available
src/models/unet_cbam.py available
src/models/cbam.py available


## 4. Original training configuration and implementation

The executable training implementation is in `train_classifier.py`, `train_segmentation.py`, and `src/train/`. These are the source files used to construct this notebook. The code cell below exposes the current script settings without writing a file. Classification trains each seed/model separately for at most 20 epochs, batch 32, Adam 1e-4, ReduceLROnPlateau on validation loss, early stopping patience 5, and saves the **minimum validation loss** checkpoint. Segmentation uses 40 epochs, batch 8, Adam 1e-4 with weight decay 1e-4, ReduceLROnPlateau on validation Dice, patience 8, and saves the **maximum validation Dice** checkpoint. Historical configuration is only partially evidenced by logs; the current script defaults do not prove every setting of the previous runs.

The following function is the notebook's executable training entry point. It calls the included original scripts, preserving their dataset, model, trainer, logger and checkpoint code. It refuses to run unless the student deliberately edits `ALLOW_LONG_TRAINING` in the final cell. It also refuses existing output paths so archived evidence is not overwritten.

In [4]:
import subprocess
import sys

def run_original_training(task, model, seed, *, allow=False):
    if not allow:
        raise RuntimeError('Long training is disabled. Explicitly opt in at the final cell.')
    if task not in ('classification', 'segmentation') or model not in ('baseline', 'cbam'):
        raise ValueError('Choose a known task and model.')
    if task == 'classification':
        script = ROOT / 'train_classifier.py'
        slug = 'baseline_resnet18' if model == 'baseline' else 'cbam_resnet18'
        outputs = [ROOT/f'checkpoints/{slug}_seed{seed}_best.pth',
                   ROOT/f'logs/{model}_seed{seed}.csv',
                   ROOT/f'results/classification_{model}_seed{seed}.csv']
    else:
        script = ROOT / 'train_segmentation.py'
        stem = f'{model}_unet_best' if seed == 42 else f'{model}_unet_seed{seed}_best'
        history = f'{model}_history' if seed == 42 else f'{model}_history_seed{seed}'
        outputs = [ROOT/f'checkpoints/{stem}.pth', ROOT/f'logs/{history}.csv']
    existing = [str(p) for p in outputs if p.exists()]
    if existing:
        raise FileExistsError('Original outputs exist; training would overwrite them: ' + ', '.join(existing))
    subprocess.run([sys.executable, str(script), '--model', model, '--seed', str(seed)],
                   cwd=ROOT, check=True)


### Full original training scripts

The following are verbatim copies of the two training entry scripts from the project, shown in Markdown so viewing or running the evidence cells cannot execute their `__main__` blocks. The executable files alongside this notebook are byte-identical copies; they import the accompanying `src/` modules for preprocessing, architectures, losses, training and evaluation. New training can only be requested through the guarded entry point above.

#### Classification training script — `train_classifier.py`

```python
import os
import time
import warnings
import argparse

import pandas as pd
import torch
from torch.cuda.amp import GradScaler
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader

os.environ.setdefault("NO_ALBUMENTATIONS_UPDATE", "1")

from src.utils.seed import set_seed
from src.datasets.classification_dataset import ClassificationDataset
from src.datasets.transforms import (
    get_train_classification_transform,
    get_val_classification_transform,
)

from src.losses.focal_loss import FocalLoss
from src.models.resnet18_classifier import ResNet18Classifier
from src.models.cbam_resnet18 import CBAMResNet18

from src.train.trainer import train_one_epoch, validate
from src.train.callbacks import EarlyStopping, CheckpointSaver
from src.train.utils import seed_everything

warnings.filterwarnings("ignore")


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    set_seed(worker_seed)


# ==========================================================
# Configuration
# ==========================================================

CONFIG = {

    "seed": 42,

    "epochs": 20,

    "batch_size": 32,

    "learning_rate": 1e-4,

    "patience": 5,

    "num_workers": 2,

    # Choose which model to train
    "model_type": "cbam",      # "baseline" or "cbam"

    # Automatically generated paths
    "checkpoint_path": None,
    "log_path": None,
    "result_path": None,
}


def configure_output_paths():
    model_slug = (
        "baseline_resnet18"
        if CONFIG["model_type"] == "baseline"
        else "cbam_resnet18"
    )

    CONFIG["checkpoint_path"] = (
        f"checkpoints/{model_slug}_seed{CONFIG['seed']}_best.pth"
    )
    CONFIG["log_path"] = f"logs/{CONFIG['model_type']}_seed{CONFIG['seed']}.csv"
    CONFIG["result_path"] = (
        f"results/classification_{CONFIG['model_type']}_seed{CONFIG['seed']}.csv"
    )


def write_seed_result(metrics_rows):
    best_row = min(metrics_rows, key=lambda row: row["val_loss"])
    peak_macro_f1_row = max(metrics_rows, key=lambda row: row["f1_macro"])
    training_time_seconds = sum(row["epoch_time"] for row in metrics_rows)

    row = {
        "model": (
            "Baseline ResNet18"
            if CONFIG["model_type"] == "baseline"
            else "CBAM-ResNet18"
        ),
        "seed": CONFIG["seed"],
        "best_epoch": best_row["epoch"],
        "selection_metric": "lowest_val_loss",
        "val_loss": best_row["val_loss"],
        "precision": best_row["precision"],
        "recall": best_row["recall"],
        "macro_f1": best_row["f1_macro"],
        "micro_f1": best_row["f1_micro"],
        "roc_auc": best_row["roc_auc"],
        "training_time_seconds": training_time_seconds,
        "training_time_minutes": training_time_seconds / 60.0,
        "epochs_completed": len(metrics_rows),
        "peak_macro_f1_epoch": peak_macro_f1_row["epoch"],
        "peak_macro_f1": peak_macro_f1_row["f1_macro"],
        "checkpoint_path": CONFIG["checkpoint_path"],
        "log_path": CONFIG["log_path"],
    }

    os.makedirs("results", exist_ok=True)

    pd.DataFrame([row]).to_csv(CONFIG["result_path"], index=False)
    print(f"\nSeed result saved to {CONFIG['result_path']}")


# ==========================================================
# Main
# ==========================================================

def main():
    
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--model",
        default=CONFIG["model_type"],
        choices=["baseline", "cbam"],
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=CONFIG["seed"],
    )

    args = parser.parse_args()

    CONFIG["model_type"] = args.model
    CONFIG["seed"] = args.seed
    configure_output_paths()

    seed_everything(CONFIG["seed"])
    
    set_seed(CONFIG["seed"])


    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print(f"\nUsing device : {device}")

    if device.type == "cuda":

        print(torch.cuda.get_device_name(0))

    os.makedirs("checkpoints", exist_ok=True)

    os.makedirs("logs", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    # -----------------------------------------
    # Read CSV
    # -----------------------------------------

    train_df = pd.read_csv(
        "data/processed/train_split.csv"
    )

    val_df = pd.read_csv(
        "data/processed/val_split.csv"
    )

    # -----------------------------------------
    # Dataset
    # -----------------------------------------

    train_dataset = ClassificationDataset(
        train_df,
        "data/train_images",
        transform=get_train_classification_transform()
    )

    val_dataset = ClassificationDataset(
        val_df,
        "data/train_images",
        transform=get_val_classification_transform()
    )

    # -----------------------------------------
    # DataLoader
    # -----------------------------------------

    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(CONFIG["seed"]),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(CONFIG["seed"]),
    )

    # -----------------------------------------
    # Model
    # -----------------------------------------

    if CONFIG["model_type"] == "baseline":

        model = ResNet18Classifier(
            num_classes=4
        )

    elif CONFIG["model_type"] == "cbam":

        model = CBAMResNet18(
            pretrained=True,
            num_classes=4
        )

    else:
        raise ValueError(f"Unknown model type: {CONFIG['model_type']}")

    model = model.to(device)

    criterion = FocalLoss()

    optimizer = Adam(
        model.parameters(),
        lr=CONFIG["learning_rate"]
    )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )

    scaler = GradScaler(
        enabled=torch.cuda.is_available()
    )

    early_stopping = EarlyStopping(
        patience=CONFIG["patience"]
    )

    saver = CheckpointSaver(
        CONFIG["checkpoint_path"]
    )

    print("\nEverything initialized successfully.")

    print(f"Train Images : {len(train_dataset)}")

    print(f"Validation Images : {len(val_dataset)}")

    print(f"Batch Size : {CONFIG['batch_size']}")

    print(f"Epochs : {CONFIG['epochs']}")

    print(f"Learning Rate : {CONFIG['learning_rate']}")
    print(f"Seed : {CONFIG['seed']}")
    print(f"Checkpoint : {CONFIG['checkpoint_path']}")
    print(f"Log : {CONFIG['log_path']}")
    print(f"Result : {CONFIG['result_path']}")

    # -------------------------------------------------
    # CSV Logger
    # -------------------------------------------------

    import csv

    with open(CONFIG["log_path"], "w", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "epoch",
            "train_loss",
            "val_loss",
            "precision",
            "recall",
            "f1_macro",
            "f1_micro",
            "roc_auc",
            "lr",
            "epoch_time"
        ])

    # -------------------------------------------------
    # Training Loop
    # -------------------------------------------------

    metrics_rows = []

    for epoch in range(CONFIG["epochs"]):

        print("\n" + "=" * 60)
        print(f"Epoch {epoch + 1}/{CONFIG['epochs']}")
        print("=" * 60)

        train_loss, epoch_time = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
        )

        val_loss, metrics = validate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        scheduler.step(val_loss)

        saver.save(model, val_loss)

        early_stopping(val_loss)

        lr = optimizer.param_groups[0]["lr"]

        print(f"Train Loss : {train_loss:.4f}")
        print(f"Val Loss   : {val_loss:.4f}")

        print(f"Precision  : {metrics['precision']:.4f}")
        print(f"Recall     : {metrics['recall']:.4f}")
        print(f"Macro F1   : {metrics['f1_macro']:.4f}")
        print(f"Micro F1   : {metrics['f1_micro']:.4f}")
        print(f"ROC-AUC    : {metrics['roc_auc']:.4f}")

        print(f"LR         : {lr:.6f}")
        print(f"Epoch Time : {epoch_time:.1f} sec")

        with open(CONFIG["log_path"], "a", newline="") as f:

            writer = csv.writer(f)

            writer.writerow([
                epoch + 1,
                train_loss,
                val_loss,
                metrics["precision"],
                metrics["recall"],
                metrics["f1_macro"],
                metrics["f1_micro"],
                metrics["roc_auc"],
                lr,
                epoch_time
            ])

        metrics_rows.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1_macro": metrics["f1_macro"],
            "f1_micro": metrics["f1_micro"],
            "roc_auc": metrics["roc_auc"],
            "lr": lr,
            "epoch_time": epoch_time,
        })

        if early_stopping.stop:

            print("\nEarly stopping triggered.")
            break

    write_seed_result(metrics_rows)

    print("\nTraining completed successfully.")
    


if __name__ == "__main__":
    main()

```

#### Segmentation training script — `train_segmentation.py`

```python
import os
import time
import random

import numpy as np
import pandas as pd

import torch
import albumentations as A

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from src.datasets.segmentation_dataset import SteelSegmentationDataset

from src.models.unet_resnet18 import UNetResNet18
from src.models.unet_cbam import UNetCBAM

from src.losses.dice_loss import DiceBCELoss

from src.train.segmentation_trainer import (
    train_one_epoch,
    validate
)

# ==========================================================
# CONFIG
# ==========================================================

CONFIG = {

    "seed": 42,

    "image_size": 256,

    "batch_size": 8,

    "epochs": 40,

    "learning_rate": 1e-4,

    "weight_decay": 1e-4,

    "patience": 8,

    "num_workers": 2,

    "pin_memory": True,

    "model_type": "cbam",      # baseline | cbam

    "train_csv":
        "data/train.csv",

    "image_dir":
        "data/train_images",

    "checkpoint_dir":
        "checkpoints",

    "log_dir":
        "logs"
}


# ==========================================================
# Seed
# ==========================================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False
    

def build_model(model_type):

    if model_type == "baseline":

        return UNetResNet18()

    elif model_type == "cbam":

        return UNetCBAM()

    raise ValueError(f"Unknown model type: {model_type}")

# ==========================================================
# Transforms
# ==========================================================

def get_train_transform():

    return A.Compose([

        A.HorizontalFlip(p=0.5),

        A.VerticalFlip(p=0.5),

        A.RandomRotate90(p=0.5)

    ])


def get_val_transform():

    return A.Compose([])

# ==========================================================
# Main
# ==========================================================

def main():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--model",
        default=CONFIG["model_type"],
        choices=["baseline", "cbam"],
        help="Model architecture: baseline or cbam"
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=CONFIG["seed"],
        help="Random seed for split and reproducibility"
    )
    args = parser.parse_args()
    CONFIG["model_type"] = args.model
    CONFIG["seed"] = args.seed

    seed_everything(CONFIG["seed"])

    device = torch.device(

        "cuda"

        if torch.cuda.is_available()

        else "cpu"

    )

    print()

    print("Using device :", device)


    if device.type == "cuda":
        print(torch.cuda.get_device_name(0))

    print()
    print("Configuration")
    print("------------------------------")
    print("Model      :", CONFIG["model_type"])
    print("Image Size :", CONFIG["image_size"])
    print("Batch Size :", CONFIG["batch_size"])
    print("Epochs     :", CONFIG["epochs"])
    print("LR         :", CONFIG["learning_rate"])
    print()

    if device.type == "cuda":

        print(torch.cuda.get_device_name(0))

    os.makedirs(

        CONFIG["checkpoint_dir"],

        exist_ok=True

    )

    os.makedirs(

        CONFIG["log_dir"],

        exist_ok=True

    )
    
    # ==========================================================
    # Load Dataset
    # ==========================================================

    master_df = pd.read_csv(

        CONFIG["train_csv"]

    )

    train_df, val_df = train_test_split(

        master_df,

        test_size=0.2,

        shuffle=True,

        random_state=CONFIG["seed"]

    )
    
    # ==========================================================
    # Datasets
    # ==========================================================

    train_transform = get_train_transform()

    val_transform = get_val_transform()

    train_dataset = SteelSegmentationDataset(

        dataframe=train_df,

        image_dir=CONFIG["image_dir"],

        transform=train_transform,

        image_size=(CONFIG["image_size"], CONFIG["image_size"])

    )

    val_dataset = SteelSegmentationDataset(

        dataframe=val_df,

        image_dir=CONFIG["image_dir"],

        transform=val_transform,

        image_size=(CONFIG["image_size"], CONFIG["image_size"])

    )
    
    # ==========================================================
    # DataLoaders
    # ==========================================================

    train_loader = DataLoader(

        train_dataset,

        batch_size=CONFIG["batch_size"],

        shuffle=True,

        num_workers=CONFIG["num_workers"],

        pin_memory=CONFIG["pin_memory"]

    )

    val_loader = DataLoader(

        val_dataset,

        batch_size=CONFIG["batch_size"],

        shuffle=False,

        num_workers=CONFIG["num_workers"],

        pin_memory=CONFIG["pin_memory"]

    )
    
    print("Everything initialized successfully.")

    print("Train Images :", len(train_dataset))

    print("Validation Images :", len(val_dataset))

    print("Batch Size :", CONFIG["batch_size"])

    print("Epochs :", CONFIG["epochs"])

    print("Learning Rate :", CONFIG["learning_rate"])

    print()
    
    # ==========================================================
    # Model
    # ==========================================================

    model = build_model(

        CONFIG["model_type"]

    )

    model = model.to(device)
    
    
    # ==========================================================
    # Loss
    # ==========================================================

    criterion = DiceBCELoss()
    
    # ==========================================================
    # Optimizer
    # ==========================================================

    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=CONFIG["learning_rate"],

        weight_decay=CONFIG["weight_decay"]

    )
    
    # ==========================================================
    # Scheduler
    # ==========================================================

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=0.5,

        patience=3

    )
    
    # ==========================================================
    # AMP
    # ==========================================================

    scaler = torch.amp.GradScaler(

        "cuda",

        enabled=device.type == "cuda"

    )
    
    history = []

    best_dice = 0.0

    patience_counter = 0
    
    for epoch in range(CONFIG["epochs"]):

        print()

        print("=" * 60)

        print(f"Epoch {epoch + 1}/{CONFIG['epochs']}")

        print("=" * 60)

        start_time = time.time()

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            device=device
        )

        val_metrics = validate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device
        )

        scheduler.step(val_metrics["dice"])

        current_lr = optimizer.param_groups[0]["lr"]

        epoch_time = time.time() - start_time

        history.append({

            "epoch": epoch + 1,

            "train_loss": train_loss,

            "val_loss": val_metrics["loss"],

            "dice": val_metrics["dice"],

            "iou": val_metrics["iou"],

            "pixel_acc": val_metrics["pixel_acc"],

            "lr": current_lr
        })
        
        if val_metrics["dice"] > best_dice:

            best_dice = val_metrics["dice"]

            patience_counter = 0

            if CONFIG["seed"] == 42:
                checkpoint_path = os.path.join(
                    CONFIG["checkpoint_dir"],
                    f"{CONFIG['model_type']}_unet_best.pth"
                )
            else:
                checkpoint_path = os.path.join(
                    CONFIG["checkpoint_dir"],
                    f"{CONFIG['model_type']}_unet_seed{CONFIG['seed']}_best.pth"
                )

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print("✓ Best model saved.")

        else:

            patience_counter += 1
            
        print(f"Train Loss : {train_loss:.4f}")

        print(f"Val Loss   : {val_metrics['loss']:.4f}")

        print(f"Dice       : {val_metrics['dice']:.4f}")

        print(f"IoU        : {val_metrics['iou']:.4f}")

        print(f"Pixel Acc  : {val_metrics['pixel_acc']:.4f}")

        print(f"LR         : {current_lr:.6f}")

        print(f"Epoch Time : {epoch_time:.1f} sec")
        
        if patience_counter >= CONFIG["patience"]:

            print()

            print("Early stopping triggered.")

            break
        
        
    history_df = pd.DataFrame(history)

    if CONFIG["seed"] == 42:
        history_path = os.path.join(
            CONFIG["log_dir"],
            f"{CONFIG['model_type']}_history.csv"
        )
    else:
        history_path = os.path.join(
            CONFIG["log_dir"],
            f"{CONFIG['model_type']}_history_seed{CONFIG['seed']}.csv"
        )
    history_df.to_csv(history_path, index=False)

    print()

    print("Training completed successfully.")
    
    
if __name__ == "__main__":

    main()
    
```

## 5. Completed training results — archived previous run

The following tables were transcribed programmatically from `logs/baseline_seed42.csv`, `logs/baseline_seed0.csv`, `logs/baseline_seed7.csv`, `logs/cbam_seed42.csv`, `logs/cbam_seed0.csv`, `logs/cbam_seed7.csv`, `logs/baseline_history.csv`, and `logs/cbam_history.csv`. They are **archived validation results from the previous run**, not notebook outputs. The classifier row at each seed is selected by lowest logged validation loss; peak F1 is an audit of the log and does not imply that peak-F1 weights were saved. Aggregate ± values use the sample standard deviation over three seeds.

| Model | Seed | Min loss epoch | Precision | Recall | Macro F1 | Micro F1 | ROC AUC | Peak F1 epoch | Peak F1 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Baseline ResNet18 | 42 | 7 | 0.8953 | 0.8000 | 0.8341 | 0.8989 | 0.9860 | 11 | 0.8598 |
| Baseline ResNet18 | 0 | 8 | 0.8340 | 0.8442 | 0.8367 | 0.8955 | 0.9870 | 12 | 0.8629 |
| Baseline ResNet18 | 7 | 8 | 0.9044 | 0.8267 | 0.8628 | 0.9120 | 0.9913 | 13 | 0.8671 |
| CBAM-ResNet18 | 42 | 9 | 0.9156 | 0.8447 | 0.8777 | 0.9128 | 0.9915 | 9 | 0.8777 |
| CBAM-ResNet18 | 0 | 8 | 0.9029 | 0.7774 | 0.8318 | 0.8933 | 0.9864 | 13 | 0.8695 |
| CBAM-ResNet18 | 7 | 12 | 0.8494 | 0.8513 | 0.8484 | 0.9080 | 0.9889 | 16 | 0.8740 |

| Model | Precision | Recall | Macro F1 | Micro F1 | ROC AUC | Peak macro F1 |
|---|---:|---:|---:|---:|---:|---:|
| Baseline ResNet18 | 0.8779 ± 0.0383 | 0.8236 ± 0.0223 | 0.8445 ± 0.0159 | 0.9021 ± 0.0087 | 0.9881 ± 0.0028 | 0.8633 ± 0.0037 |
| CBAM-ResNet18 | 0.8893 ± 0.0352 | 0.8245 ± 0.0409 | 0.8526 ± 0.0232 | 0.9047 ± 0.0102 | 0.9889 ± 0.0026 | 0.8737 ± 0.0041 |

Segmentation final epoch and best Dice are separate archived validation observations. The saved checkpoint policy is best Dice, although the checkpoint files are missing here.

| Model | Final epoch | Val loss | Dice | IoU | Pixel accuracy | Best Dice epoch | Best Dice |
|---|---:|---:|---:|---:|---:|---:|---:|
| Baseline U-Net | 40 | 0.1093 | 0.8372 | 0.8114 | 0.9896 | 40 | 0.8372 |
| CBAM U-Net | 40 | 0.1089 | 0.8368 | 0.8116 | 0.9905 | 38 | 0.8393 |

In [5]:
# Read-only inventory of the original archived histories; no metrics are recalculated.
def csv_rows(rel):
    with (ROOT / rel).open(newline='', encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))

for rel in [
    'logs/baseline_seed42.csv', 'logs/baseline_seed0.csv', 'logs/baseline_seed7.csv',
    'logs/cbam_seed42.csv', 'logs/cbam_seed0.csv', 'logs/cbam_seed7.csv',
    'logs/baseline_history.csv', 'logs/cbam_history.csv',
]:
    print(rel, len(csv_rows(rel)), 'archived epoch rows')


logs/baseline_seed42.csv 12 archived epoch rows
logs/baseline_seed0.csv 13 archived epoch rows
logs/baseline_seed7.csv 13 archived epoch rows
logs/cbam_seed42.csv 14 archived epoch rows
logs/cbam_seed0.csv 13 archived epoch rows
logs/cbam_seed7.csv 17 archived epoch rows
logs/baseline_history.csv 40 archived epoch rows
logs/cbam_history.csv 40 archived epoch rows


### Saved figures from the previous run

These are original files copied from `figures/`. They are shown as archived artifacts; the notebook has not generated or re-executed them. Figure 4's class counts are only reported counts until original label files are supplied.

![Reported class distribution](figures/fig4_class_distribution.png)

![Classification macro F1 comparison](figures/fig6_macro_f1_comparison.png)

![Segmentation validation Dice by epoch](figures/fig7_dice_vs_epoch.png)

![Classification training and validation losses](figures/fig8_classification_train_val_loss.png)

![Segmentation training and validation losses](figures/fig9_segmentation_train_val_loss.png)

## 6. Evaluation using existing checkpoints — optional, read only

No checkpoint or image data is in this checkout. The original scripts evaluate **validation**, and no independent test metrics or predictions were saved. If original files are restored, the next cells can evaluate a classifier checkpoint on `data/processed/test_split.csv` (new independent test result) or, if explicitly changed, on the original validation CSV. They use `ClassificationDataset`, the deterministic validation transform, `FocalLoss`, and `src.train.trainer.validate`. Any newly computed score must be reported separately from the archived validation tables. Model construction uses `pretrained=False` because the checkpoint supplies weights and a network download is unnecessary. The cell prints unavailable and exits harmlessly when requirements are absent.

In [ ]:
# Quick classifier evaluation. Read-only; never calls an optimizer or trainer.
CHECKPOINT = ROOT/'checkpoints/cbam_resnet18_seed42_best.pth'
TEST_CSV = ROOT/'data/processed/test_split.csv'
IMAGE_DIR = ROOT/'data/train_images'
if not (CHECKPOINT.exists() and TEST_CSV.exists() and IMAGE_DIR.exists()):
    print('Unavailable: supply the original CBAM seed-42 checkpoint, test_split.csv, and train_images.')
else:
    import pandas as pd
    import torch
    from torch.utils.data import DataLoader
    from src.datasets.classification_dataset import ClassificationDataset
    from src.datasets.transforms import get_val_classification_transform
    from src.models.cbam_resnet18 import CBAMResNet18
    from src.losses.focal_loss import FocalLoss
    from src.train.trainer import validate
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    df_test = pd.read_csv(TEST_CSV)
    test_loader = DataLoader(ClassificationDataset(df_test, str(IMAGE_DIR),
                         transform=get_val_classification_transform()),
                         batch_size=32, shuffle=False, num_workers=0)
    model = CBAMResNet18(pretrained=False, num_classes=4).to(device)
    state = torch.load(CHECKPOINT, map_location=device, weights_only=True)
    model.load_state_dict(state, strict=True)
    test_loss, test_metrics = validate(model, test_loader, FocalLoss(), device)
    print('NEW TEST evaluation, separate from archived validation results:')
    print('loss', test_loss, test_metrics)


The original segmentation evaluation has no independent test split. Its code reconstructs the **seed-42 annotation-row validation split** from `data/train.csv`; that split may contain the same image ID on both sides. The cell below follows the original `SteelSegmentationDataset`, no-op validation transform, `DiceBCELoss`, and `src.train.segmentation_trainer.validate`. It is optional because both data and weights are absent. The CBAM U-Net constructor in the original source requests pretrained ResNet18 weights before loading its checkpoint, which may require the torchvision cache or network; no source code was changed to hide that limitation.

In [ ]:
# Quick segmentation validation, original procedure. Read-only.
SEG_CHECKPOINT = ROOT/'checkpoints/baseline_unet_best.pth'
ANNOTATIONS = ROOT/'data/train.csv'
if not (SEG_CHECKPOINT.exists() and ANNOTATIONS.exists() and IMAGE_DIR.exists()):
    print('Unavailable: supply baseline_unet_best.pth, data/train.csv, and train_images.')
else:
    import albumentations as A
    import pandas as pd
    import torch
    from sklearn.model_selection import train_test_split
    from torch.utils.data import DataLoader
    from src.datasets.segmentation_dataset import SteelSegmentationDataset
    from src.models.unet_resnet18 import UNetResNet18
    from src.losses.dice_loss import DiceBCELoss
    from src.train.segmentation_trainer import validate as validate_segmentation
    _, seg_val_df = train_test_split(pd.read_csv(ANNOTATIONS), test_size=0.2,
                                     shuffle=True, random_state=42)
    seg_loader = DataLoader(SteelSegmentationDataset(seg_val_df, str(IMAGE_DIR),
                            transform=A.Compose([]), image_size=(256, 256)),
                            batch_size=8, shuffle=False, num_workers=0)
    seg_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    seg_model = UNetResNet18(pretrained=False).to(seg_device)
    seg_model.load_state_dict(torch.load(SEG_CHECKPOINT, map_location=seg_device,
                                         weights_only=True), strict=True)
    print('NEW validation replay, separate from archived metrics:')
    print(validate_segmentation(seg_model, seg_loader, DiceBCELoss(), seg_device))


## 7. Archived sample predictions and error inspection

The original run saved selected example figures and manifests, but no full per-image prediction table. These examples are **selected previous-run outputs**, not fresh inference and not representative error rates. The Class 2 false negative appears in both models' inspected examples (`results/error_analysis/classification/classification_error_manifest.csv`). The segmentation comparison is an inspected image-level foreground union, so its displayed Dice is defined differently from the validation table's image–class average.

![Baseline Class 2 false negative](results/error_analysis/classification/baseline_class2_false_negative_1.png)

![CBAM Class 2 false negative](results/error_analysis/classification/cbam_class2_false_negative_1.png)

![Archived Grad-CAM classification example](results/gradcam/classification/example_01_gradcam.png)

![Archived CBAM attention example](results/attention/cbam/example_01_cbam_attention.png)

![Archived segmentation comparison](results/explainability/segmentation/example_01_comparison.png)

In [6]:
# Inspect the saved example metadata without generating new predictions.
for rel in ['results/error_analysis/classification/classification_error_manifest.csv',
            'results/gradcam/classification/gradcam_manifest.csv',
            'results/explainability/segmentation/segmentation_explainability_manifest.csv']:
    records = csv_rows(rel)
    print(rel, len(records), 'archived records; first:', records[0] if records else 'none')


results/error_analysis/classification/classification_error_manifest.csv 32 archived records; first: {'model': 'Baseline ResNet18', 'class_name': 'Class 1', 'class_index': '1', 'error_type': 'false_negative', 'rank': '1', 'image_id': 'fc0541a53.jpg', 'true_label': '1', 'predicted_label': '0', 'probability': '0.059403978288173676', 'figure_path': 'results/error_analysis/classification/baseline_class1_false_negative_1.png'}
results/gradcam/classification/gradcam_manifest.csv 6 archived records; first: {'image_id': '000a4bcdd.jpg', 'target_class': 'Class 1', 'baseline_probability': '0.940416', 'cbam_probability': '0.689529', 'figure_path': 'results/gradcam/classification/example_01_gradcam.png'}
results/explainability/segmentation/segmentation_explainability_manifest.csv 18 archived records; first: {'image_id': '005f19695.jpg', 'model': 'baseline', 'dice': '0.748603', 'true_positive_pixels': '4020', 'false_negative_pixels': '384', 'false_positive_pixels': '2316', 'figure_path': 'results/ex

## 8. Method and evidence limits

All numerical performance tables above are **archived previous-run validation results** from the included raw CSV histories. The classification table selects the minimum-validation-loss row in each of six logs; peak macro F1 is a retrospective log audit and does not establish a separately saved peak-F1 checkpoint. The segmentation table separates epoch-40 metrics from maximum-Dice epochs. The raw final Dice values are 0.8371536101807248 (baseline) and 0.836827642199668 (CBAM), a difference of 0.0003259679810568; subtracting displayed four-decimal values gives 0.0004.

The current classification script uses `ReduceLROnPlateau`, and segmentation keeps four mask channels. The U-Net variants also differ at final upsampling, so the single-run segmentation comparison cannot isolate CBAM. Original image data, splits, checkpoints, run configuration and independent test metrics are unavailable; the reported image counts cannot be independently audited here. No historical results were recomputed or replaced by a new evaluation.

## 9. Interpretation and submission status

The archived minimum-loss classifier comparison favours CBAM by 0.0081 macro F1 in the three-seed mean, with larger observed SD (0.0232 versus 0.0159). This is a validation comparison, not a statistical or independent-test claim. Peak-F1 selection changes both models' summaries and is optimistic because selection and reporting use the same validation histories. Single-run segmentation final Dice differs only slightly and the models also have different output heads. Selected error images show examples worth reviewing, not population failure rates.

**Unavailable for verification:** original image data; `data/train.csv`; `data/processed/master_labels.csv`, `data/processed/train_split.csv`, `data/processed/val_split.csv`, `data/processed/test_split.csv`; `checkpoints/*.pth`; independent test predictions/metrics; full run configuration/dependency lock; measured runtime CSV. Supply the exact original data and checkpoint files rather than regenerating them as if they were the prior run. For reproducibility, supply the exact original dataset, split CSVs and checkpoints listed in README.md; do not treat new evaluation as part of the archived run.

## 10. Deliberate optional training action — do not run for this submission

This final cell is the only training call. Its default flag is false. To run a **new** experiment in a separate copy with original data and a compatible PyTorch environment, change the flag and specify one task/model/seed. The helper refuses paths that would overwrite copied logs or existing checkpoints. A new run is not evidence for the historical report numbers.

In [ ]:
ALLOW_LONG_TRAINING = False  # Deliberately edit to True only for a separate new experiment.
if ALLOW_LONG_TRAINING:
    run_original_training('classification', 'cbam', 42, allow=True)
else:
    print('Training disabled; archived results remain unchanged.')
